In [ ]:
# Cell 1 - Setup leve do Colab e bootstrap do repositório
import io
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

SETUP_VERSION = "2026-05-01-v1"
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    pass

REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = Path("/content/TCC")
BRANCH = os.environ.get("NVS_BENCHMARK_BRANCH", "update")
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()


def run_cmd(cmd, *, check: bool = False, capture: bool = False):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, text=True, check=check, capture_output=capture)


def _repo_base(repo_url: str) -> str:
    return repo_url[:-4] if repo_url.endswith(".git") else repo_url


def _repo_slug(repo_url: str) -> str:
    return "/".join(_repo_base(repo_url).rstrip("/").split("/")[-2:])


def _clone_url(repo_url: str) -> str:
    if not GITHUB_TOKEN:
        return repo_url
    return _repo_base(repo_url).replace("https://", f"https://{GITHUB_TOKEN}@") + ".git"


def ensure_repo() -> None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if run_cmd(["git", "clone", "--depth", "1", "--branch", BRANCH, _clone_url(REPO_URL), str(REPO_DIR)]).returncode == 0:
        return
    zip_url = f"https://codeload.github.com/{_repo_slug(REPO_URL)}/zip/refs/heads/{BRANCH}"
    print("Fallback ZIP:", zip_url)
    data = urllib.request.urlopen(urllib.request.Request(zip_url)).read()
    tmp_dir = REPO_DIR.parent / "_repo_tmp"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(io.BytesIO(data)).extractall(tmp_dir)
    extracted = next((item for item in tmp_dir.iterdir() if item.is_dir()), None)
    if extracted is None:
        raise RuntimeError("Repositorio nao encontrado no ZIP baixado")
    extracted.rename(REPO_DIR)
    shutil.rmtree(tmp_dir, ignore_errors=True)


if IN_COLAB:
    ensure_repo()
    os.chdir(REPO_DIR)
    run_cmd([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
else:
    print("Nao detectado Colab; usando o workspace atual.")

SETUP_VERSION: 2026-04-29-v2
Nao detectado Colab. Ajuste os caminhos para execucao local.


In [ ]:
# Cell 2 - Configuracao da rodada
from pathlib import Path

RUN_ID = "colab_full_matrix"
PRESET = "quick"
RUN_MODE = "full"  # full | quick_check
QUICK_CHECK_MAX_COMBOS = 2
ONLY_DATASETS = None
ONLY_METHODS = None
SKIP_METHODS = []
STRICT_RESULTS = True
RUN_DATASET_INSTALL = False
RUN_METHOD_INSTALL = False
GENERATE_PDF = False
APPLY_COMPATIBILITY_FILTER = True
VERBOSE_MODE = True
VERBOSE_HEARTBEAT_SECONDS = 30

DATA_DIR = Path("./data")
ARTIFACTS_DIR = Path("./artifacts")
METRICS_DIR = ARTIFACTS_DIR / "metrics"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
LOG_DIR = Path("./logs")
RUNS_DIR = METRICS_DIR / "matrix_runs"

PLAN_FILE = METRICS_DIR / f"{RUN_ID}_plan.json"
EXECUTED_FILE = METRICS_DIR / f"{RUN_ID}_executed.json"
SKIPPED_FILE = METRICS_DIR / f"{RUN_ID}_skipped.json"
STATUS_FILE = METRICS_DIR / f"{RUN_ID}_status.json"
SNAPSHOT_FILE = METRICS_DIR / f"{RUN_ID}.json"
REPORT_NAME = f"{RUN_ID}_report"

In [ ]:
# Cell 3 - Imports, registry e utilitarios
import hashlib
import time
from dataclasses import asdict

import torch

from nvs_benchmark.cli_extensions import validate_dataset_integrity_preflight
from nvs_benchmark.core import DatasetSpec, RunConfig
from nvs_benchmark.data import SUPPORTED_DATASETS, load_dataset, validate_dataset
from nvs_benchmark.methods import build_registry_with_all_methods


def ensure_imports() -> None:
    src_dir = (REPO_DIR / "src") if IN_COLAB else Path("./src")
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    import nvs_benchmark  # noqa: F401


ensure_imports()
registry = build_registry_with_all_methods()
METHOD_IDS = registry.list_ids()
print("Metodos:", METHOD_IDS)
print("CUDA disponivel:", torch.cuda.is_available())

METHOD_REPOS = {
    "nerf_static": ("https://github.com/albertpumarola/D-NeRF.git", "main", "third_party/d_nerf"),
    "nerf_dynamic": ("https://github.com/albertpumarola/D-NeRF.git", "main", "third_party/d_nerf"),
    "gs_static": ("https://github.com/graphdeco-inria/gaussian-splatting.git", "main", "third_party/gaussian_splatting"),
}


def dump_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def unique_paths(paths):
    return sorted({p.resolve() for p in paths if p and p.exists()})


def scene_id_for(root: Path, base: Path) -> str:
    try:
        rel = root.resolve().relative_to(base.resolve())
        return "__".join(rel.parts) if rel.parts else root.name
    except Exception:
        return root.name


def scene_roots(dataset: str) -> list[Path]:
    base = DATA_DIR / dataset
    if not base.exists():
        return []
    roots: list[Path] = []
    if dataset in {"blender_synthetic", "d_nerf"}:
        roots.extend(item.parent for item in base.rglob("transforms_train.json"))
    elif dataset == "mipnerf360":
        roots.extend(item.parent for item in base.rglob("poses_bounds.npy"))
        roots.extend(item.parent for item in base.rglob("transforms_train.json"))
        roots.extend(item.parent for item in base.rglob("sparse/0") if item.is_dir())
        roots.extend(item.parent for item in base.rglob("images") if item.is_dir())
    elif dataset == "tanks_and_temples":
        roots.extend(item.parent for item in base.rglob("transforms_train.json"))
        roots.extend(item.parent for item in base.rglob("poses_bounds.npy"))
        roots.extend(item.parent for item in base.rglob("images") if item.is_dir())
    else:
        candidates = [base] + [item for item in base.rglob("*") if item.is_dir()]
        for item in candidates:
            valid, _ = validate_dataset("custom", item)
            if valid:
                roots.append(item)
    return unique_paths(roots)


def preflight_dataset(dataset: str, root: Path) -> tuple[bool, str | DatasetSpec]:
    valid, message = validate_dataset(dataset, root)
    if not valid:
        return False, message
    try:
        return True, load_dataset(dataset, root)
    except Exception as exc:
        return False, str(exc)


def method_supports(dataset: str, method_id: str) -> bool:
    method = registry.get(method_id)
    config = RunConfig(
        run_id="preflight",
        dataset=DatasetSpec(name=dataset, root="."),
        method=method_id,
        output_dir=str(ARTIFACTS_DIR),
        log_dir=str(LOG_DIR),
    )
    method.validate_config(config)
    return True

Diretorio atual: c:\Users\Admin\Projetos\TCC\notebooks
Python: c:\Users\Admin\Projetos\TCC\venv\Scripts\python.exe
CUDA disponivel: False
Imports principais validados.


In [ ]:
# Cell 4 - Instalacao opcional e preflight leve
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if RUN_DATASET_INSTALL:
    run_cmd([sys.executable, "-m", "nvs_benchmark.cli", "install", "--catalog-file", "./configs/install_catalog.json", "--only", "datasets", "--execute"], check=False)

if RUN_METHOD_INSTALL:
    for method_id, (repo_url, branch, repo_path) in METHOD_REPOS.items():
        target = Path(repo_path)
        if target.exists():
            continue
        if target.parent.exists() is False:
            target.parent.mkdir(parents=True, exist_ok=True)
        if run_cmd(["git", "clone", "--depth", "1", "--branch", branch, repo_url, repo_path], capture=True).returncode != 0:
            fallback = f"https://codeload.github.com/{repo_url.split('github.com/', 1)[-1].removesuffix('.git')}/zip/refs/heads/{branch}"
            print("Fallback ZIP:", fallback)
            data = urllib.request.urlopen(urllib.request.Request(fallback)).read()
            tmp_dir = Path(repo_path).parent / "_method_tmp"
            if tmp_dir.exists():
                shutil.rmtree(tmp_dir)
            tmp_dir.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(io.BytesIO(data)).extractall(tmp_dir)
            extracted = next((item for item in tmp_dir.iterdir() if item.is_dir()), None)
            if extracted is None:
                raise RuntimeError(f"Falha ao instalar repo do metodo {method_id}")
            if Path(repo_path).exists():
                shutil.rmtree(repo_path)
            extracted.rename(repo_path)
            shutil.rmtree(tmp_dir, ignore_errors=True)

print("Ambiente pronto.")

Sistema Operacional: Windows

Datasets Disponiveis (0):

Para instalar manualmente, ajuste ONLY_DATASETS e execute a Celula 4.5 abaixo.


In [25]:
# Celula 4.5 - Instalar datasets selecionados (cross-platform)
# Edite DATASETS_TO_INSTALL para escolher quais baixar
# Preenchido com todos os datasets listados em configs/install_catalog.json
DATASETS_TO_INSTALL = ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples']  # ex.: ['blender_synthetic', 'd_nerf']
EXECUTE_INSTALL = False  # Mude para True para executar de verdade

if not catalog_datasets:
    raise RuntimeError("catalog_datasets vazio. Rode a Celula 4 antes da 4.5.")

if not DATASETS_TO_INSTALL:
    print("Nenhum dataset selecionado em DATASETS_TO_INSTALL.")
    print("")
    print("Para instalar, edite DATASETS_TO_INSTALL com os nomes desejados, ex.:")
    print("  DATASETS_TO_INSTALL = ['blender_synthetic']")
    print("")
    print("Depois mude EXECUTE_INSTALL = True e rode a celula novamente.")
else:
    selected_to_install = [d for d in catalog_datasets if d['id'] in DATASETS_TO_INSTALL]

    print("")
    print(f"Instalacao de {len(selected_to_install)} dataset(s) (OS: {OS_NAME})")
    print("=" * 70)

    for ds in selected_to_install:
        msg = _install_dataset(ds, execute=EXECUTE_INSTALL)
        print(msg)

    print("=" * 70)
    if EXECUTE_INSTALL:
        print("Instalacao concluida. Rode a Celula 5 para descobrir datasets instalados.")
    else:
        print("Modo de planejamento. Mude EXECUTE_INSTALL = True para instalar de verdade.")


Instalacao de 4 dataset(s) (OS: Windows)
[plan] blender_synthetic: seria instalado em ./data/blender_synthetic/nerf_synthetic/lego
[plan] d_nerf: seria instalado em ./data/d_nerf
[plan] mipnerf360: seria instalado em ./data/mipnerf360
[plan] tanks_and_temples: seria instalado em ./data/tanks_and_temples
Modo de planejamento. Mude EXECUTE_INSTALL = True para instalar de verdade.


In [27]:
# Celula 4.6 - Detectar e corrigir estrutura incorreta de datasets
import shutil

# Mapeamento de cenas para datasets
DATASET_SCENES = {
    "d_nerf": {"bicycle", "bonsai", "horsepower", "jumping_jacks", "magic_carpet", "mutant", "standup", "underfitting"},
    "mipnerf360": {"bicycle", "bonsai", "counter", "garden", "kitchen", "room", "stump"},
}

def detect_misplaced_scenes(data_root: Path) -> dict:
    """Detecta cenas em locais errados e retorna mapa de correções."""
    misplaced = {}
    
    for dataset_name, scenes in DATASET_SCENES.items():
        for item in data_root.iterdir():
            if not item.is_dir() or item.name == dataset_name:
                continue
            
            if item.name in scenes:
                if dataset_name not in misplaced:
                    misplaced[dataset_name] = []
                misplaced[dataset_name].append(item.name)
    
    return misplaced

def fix_misplaced_structure(data_root: Path, dry_run: bool = True) -> tuple[int, list]:
    """Reorganiza estrutura incorreta. Retorna (count, messages)."""
    misplaced = detect_misplaced_scenes(data_root)
    
    if not misplaced:
        return 0, ["[ok] Estrutura de datasets ja esta correta"]
    
    messages = []
    count = 0
    
    for dataset_name, scenes in misplaced.items():
        dataset_root = data_root / dataset_name
        
        for scene in scenes:
            src = data_root / scene
            dst = dataset_root / scene
            
            if not src.exists():
                continue
            
            if dry_run:
                messages.append(f"[plan] {scene} -> {dataset_name}/")
            else:
                dataset_root.mkdir(parents=True, exist_ok=True)
                if dst.exists():
                    shutil.rmtree(dst, ignore_errors=True)
                shutil.move(str(src), str(dst))
                messages.append(f"[ok] {scene} movido para {dataset_name}/")
                count += 1
    
    # Limpa arquivos de configuração deixados por downloads
    temp_files = {"download_t2_dataset.py", "flowers.txt", "treehill.txt"}
    for fname in temp_files:
        fpath = data_root / fname
        if fpath.exists() and not dry_run:
            fpath.unlink()
            messages.append(f"[cleanup] Removido arquivo temporario {fname}")
            count += 1
    
    return count, messages

# Executa diagnostico automatico
data_root = Path("./data")
if data_root.exists():
    misplaced = detect_misplaced_scenes(data_root)
    
    print("Diagnostico de estrutura de datasets:")
    print("=" * 70)
    
    if not misplaced:
        print("[ok] Estrutura de datasets ja esta correta!")
    else:
        print("[aviso] Estrutura incorreta detectada:")
        for dataset_name, scenes in misplaced.items():
            print(f"  {dataset_name}: {', '.join(sorted(scenes))}")
        
        print("")
        print("Para corrigir, execute as linhas abaixo:")
        print("=" * 70)
        print("# Descomente a linha abaixo e execute para CORRIGIR a estrutura")
        print("# count, msgs = fix_misplaced_structure(Path('./data'), dry_run=False)")
        print("# for msg in msgs: print(msg)")
        print("")
        print("Modo teste (sem fazer mudanças):")
        count, msgs = fix_misplaced_structure(Path("./data"), dry_run=True)
        for msg in msgs:
            print(msg)
        print(f"Total de acoes propostas: {count}")
else:
    print("[info] Diretorio ./data nao existe ainda")


[info] Diretorio ./data nao existe ainda


In [ ]:
# Celula 4.7 - Limpar datasets para re-extrair (opcional)
import shutil

DATASETS_TO_CLEAN = []  # ex.: ["d_nerf", "mipnerf360", "tanks_and_temples"]

if DATASETS_TO_CLEAN:
    data_root = Path("./data")
    print(f"Limpando {len(DATASETS_TO_CLEAN)} dataset(s)...")
    print("=" * 70)
    
    for dataset_name in DATASETS_TO_CLEAN:
        dataset_path = data_root / dataset_name
        
        if dataset_path.exists():
            shutil.rmtree(dataset_path, ignore_errors=True)
            print(f"[ok] Removido: {dataset_name}/")
        else:
            print(f"[skip] Nao existe: {dataset_name}/")
    
    print("=" * 70)
    print("Datasets removidos. Rode Celula 4.5 com EXECUTE_INSTALL=True para re-extrair.")
else:
    print("[info] Nenhum dataset selecionado para limpeza.")
    print("Para limpar, edite DATASETS_TO_CLEAN, ex.:")
    print("  DATASETS_TO_CLEAN = ['d_nerf', 'mipnerf360']")


In [ ]:
# Cell 5 - Descoberta de cenas e plano unico por cena/método
selected_datasets = [item for item in (ONLY_DATASETS or SUPPORTED_DATASETS) if item in SUPPORTED_DATASETS]
selected_methods = [item for item in (ONLY_METHODS or METHOD_IDS) if item in METHOD_IDS and item not in set(SKIP_METHODS)]

method_families = {
    "nerf_static": {"blender_synthetic", "d_nerf", "custom"},
    "nerf_dynamic": {"d_nerf", "custom"},
    "gs_static": {"blender_synthetic", "mipnerf360", "tanks_and_temples", "custom"},
    "gs_dynamic": {"d_nerf", "custom"},
}

matrix_plan: list[dict] = []
skipped: list[dict] = []
seen_keys: set[str] = set()

for dataset in selected_datasets:
    roots = scene_roots(dataset)
    if not roots:
        skipped.append({"dataset": dataset, "scene_id": None, "method": None, "reason": "dataset nao encontrado"})
        continue
    for root in roots:
        scene_id = scene_id_for(root, DATA_DIR / dataset)
        ok, payload = preflight_dataset(dataset, root)
        if not ok:
            skipped.append({"dataset": dataset, "scene_id": scene_id, "method": None, "root": str(root), "reason": payload})
            continue
        dataset_spec = payload
        for method_id in selected_methods:
            combo_key = f"{dataset}__{scene_id}__{method_id}"
            if combo_key in seen_keys:
                continue
            seen_keys.add(combo_key)
            if APPLY_COMPATIBILITY_FILTER and dataset not in method_families.get(method_id, {dataset}):
                skipped.append({"dataset": dataset, "scene_id": scene_id, "method": method_id, "root": str(root), "reason": "incompatibilidade dataset/metodo"})
                continue
            try:
                config = RunConfig(
                    run_id=f"{RUN_ID}__{combo_key}",
                    dataset=dataset_spec,
                    method=method_id,
                    output_dir=str(ARTIFACTS_DIR),
                    log_dir=str(LOG_DIR),
                    extra={"preset": PRESET, "scene_id": scene_id, "matrix_key": combo_key},
                )
                registry.get(method_id).validate_config(config)
            except Exception as exc:
                skipped.append({"dataset": dataset, "scene_id": scene_id, "method": method_id, "root": str(root), "reason": str(exc)})
                continue
            matrix_plan.append({
                "dataset": dataset,
                "scene_id": scene_id,
                "root": str(root),
                "method": method_id,
                "split": dataset_spec.split,
                "matrix_key": combo_key,
            })

if RUN_MODE == "quick_check":
    matrix_plan = matrix_plan[:QUICK_CHECK_MAX_COMBOS]

status_summary = {
    "datasets_selected": selected_datasets,
    "methods_selected": selected_methods,
    "plan_count": len(matrix_plan),
    "skip_count": len(skipped),
    "run_mode": RUN_MODE,
    "compatibility_filter": APPLY_COMPATIBILITY_FILTER,
}

dump_json(PLAN_FILE, matrix_plan)
dump_json(SKIPPED_FILE, skipped)
dump_json(STATUS_FILE, status_summary)

print("=" * 70)
print("Plano da matriz")
print("=" * 70)
print(f"Datasets selecionados: {selected_datasets}")
print(f"Metodos selecionados: {selected_methods}")
print(f"Combinacoes planejadas: {len(matrix_plan)}")
print(f"Combinacoes puladas: {len(skipped)}")
for item in skipped[:10]:
    print(f"[skip] {item.get('dataset')} / {item.get('scene_id')} / {item.get('method')}: {item.get('reason')}")
if len(skipped) > 10:
    print(f"... ({len(skipped) - 10} skips adicionais)")

Datasets suportados: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Datasets selecionados: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Metodos selecionados: ['gs_dynamic', 'gs_static', 'nerf_dynamic', 'nerf_static']

Raiz de descoberta: C:\Users\Admin\Projetos\TCC
Diagnostico de Datasets:
[ok] blender_synthetic              encontrado em C:\Users\Admin\Projetos\TCC\data\blender_synthetic\nerf_synthetic\lego
[x]  d_nerf                         NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\d_nerf nao existe)
[x]  mipnerf360                     NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\mipnerf360 nao existe)
[x]  tanks_and_temples              NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\tanks_and_temples nao existe)
[x]  custom                         NAO encontrado (pasta C:\Users\Admin\Projetos\TCC\data\custom nao existe)



In [ ]:
# Celula 5.2 - Setup third_party repos (dependencies)
import io as io_module

third_party_repos = {
    "gaussian_splatting": {
        "url": "https://github.com/graphdeco-inria/gaussian-splatting.git",
        "branch": "main",
        "zip_url": "https://codeload.github.com/graphdeco-inria/gaussian-splatting/zip/refs/heads/main",
    },
    "d_nerf": {
        "url": "https://github.com/albertpumarola/D-NeRF.git",
        "branch": "main",
        "zip_url": "https://codeload.github.com/albertpumarola/D-NeRF/zip/refs/heads/main",
    },
    "nerf": {
        "url": "https://github.com/bmild/nerf.git",
        "branch": "master",
        "zip_url": "https://codeload.github.com/bmild/nerf/zip/refs/heads/master",
    },
}

third_party_dir = Path("./third_party")
third_party_dir.mkdir(exist_ok=True)

def clone_third_party_via_git(repo_name, url, branch, repo_path, retries=3):
    """Tenta clonar repo via git."""
    for attempt in range(1, retries + 1):
        print(f"  [git tentativa {attempt}/{retries}] ...", end=" ", flush=True)
        
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        
        cmd = [
            "git", "clone", "--depth", "1",
            "--branch", branch,
            url,
            str(repo_path)
        ]
        
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if proc.returncode == 0:
                print("OK")
                return True
            else:
                print(f"falhou (code={proc.returncode})")
        except subprocess.TimeoutExpired:
            print("timeout")
        except Exception as e:
            print(f"erro ({e})")
    
    return False

def clone_third_party_via_zip(repo_name, zip_url, repo_path, expected_prefix):
    """Tenta baixar repo via ZIP como fallback."""
    print(f"  [zip fallback] ...", end=" ", flush=True)
    
    try:
        if repo_path.exists():
            shutil.rmtree(repo_path, ignore_errors=True)
        
        req = urllib.request.Request(zip_url)
        data = urllib.request.urlopen(req, timeout=60).read()
        
        zf = zipfile.ZipFile(io_module.BytesIO(data))
        zf.extractall(third_party_dir)
        
        # Procura pasta extraida e renomeia
        extracted = None
        for p in third_party_dir.iterdir():
            if p.is_dir() and p.name.startswith(expected_prefix):
                extracted = p
                break
        
        if extracted is None:
            print(f"falhou (pasta {expected_prefix}* nao encontrada)")
            return False
        
        extracted.rename(repo_path)
        print("OK")
        return True
    
    except Exception as e:
        print(f"falhou ({str(e)[:50]})")
        return False

print("=" * 70)
print("Verificando third_party repos:")
print("=" * 70)

for repo_name, repo_info in third_party_repos.items():
    repo_path = third_party_dir / repo_name
    
    # Check if already exists with .git
    if repo_path.exists() and (repo_path / ".git").exists():
        print(f"✓ {repo_name:25s} ja existe (git)")
        continue
    
    if repo_path.exists() and not (repo_path / ".git").exists():
        print(f"⚠ {repo_name:25s} existe mas sem .git (removendo...)")
        shutil.rmtree(repo_path, ignore_errors=True)
    
    print(f"⏳ {repo_name:25s}", end=" ")
    
    # Try git clone first
    if clone_third_party_via_git(
        repo_name,
        repo_info["url"],
        repo_info["branch"],
        repo_path
    ):
        print(f"✓ {repo_name:25s} clonado com git")
    # Fallback to ZIP
    elif clone_third_party_via_zip(
        repo_name,
        repo_info["zip_url"],
        repo_path,
        repo_name
    ):
        print(f"✓ {repo_name:25s} extraido via zip")
    else:
        print(f"✗ {repo_name:25s} FALHA permanente")
        print(f"    Tentativas: git clone + zip download")
        print(f"    Verifique conectividade ou firewall")

print("=" * 70)


In [ ]:
# Celula 5.1 - Diagnostico: Verificar third_party existente
third_party_path = Path("./third_party")

print("=" * 70)
print("Diagnostico de third_party/:")
print("=" * 70)

if third_party_path.exists():
    dirs = [d.name for d in third_party_path.iterdir() if d.is_dir()]
    print(f"Encontrado {len(dirs)} diretorios em ./third_party/:")
    for d in dirs:
        git_ok = "✓ git" if (third_party_path / d / ".git").exists() else "✗ sem git"
        print(f"  - {d:30s} ({git_ok})")
else:
    print("Diretorio ./third_party/ nao existe ainda.")

print("")
print("Se todas as dependencias ja estao presentes, pode pular a celula 5.2.")
print("Se faltarem, a celula 5.2 fara clone automaticamente.")
print("=" * 70)
print("")


In [ ]:
# Celula 5.5 - Diagnostico: Procurar transforms_train.json em todo ./data
import os

print("Busca completa por transforms_train.json em ./data/:")
print("=" * 70)

data_path = Path("./data")
if not data_path.exists():
    print("[error] Diretorio ./data nao existe")
else:
    found_any = False
    for item in data_path.rglob("transforms_train.json"):
        found_any = True
        print(f"✓ {item}")

    if not found_any:
        print("[nenhum] Nenhum transforms_train.json encontrado em ./data/")

    print("")
    print("Estrutura de ./data/ (ate 2 niveis):")
    print("=" * 70)
    for root, dirs, files in os.walk("./data", topdown=True):
        level = root.replace("./data", "").count(os.sep)
        if level > 2:
            dirs[:] = []
            continue

        indent = " " * (2 * level)
        folder_name = os.path.basename(root) or "data"
        print(f"{indent}{folder_name}/")

        subindent = " " * (2 * (level + 1))
        for file_name in sorted(files)[:10]:
            print(f"{subindent}{file_name}")
        if len(files) > 10:
            print(f"{subindent}... ({len(files) - 10} mais arquivos)")

    print("")
    print("Se ainda nao encontrar datasets, rode novamente Cell 4.5 com EXECUTE_INSTALL=True")

In [5]:
# Celula 5.6 - Mapear scene roots por dataset
from collections import defaultdict

try:
    dataset_ids = [d.item_id for d in catalog.datasets]
except Exception:
    try:
        from nvs_benchmark.install import load_install_catalog
        _catalog = load_install_catalog("./configs/install_catalog.json")
        dataset_ids = [d.item_id for d in _catalog.datasets]
    except Exception:
        dataset_ids = ["blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples", "custom"]

scene_roots = defaultdict(set)
all_transforms = sorted(Path("./data").rglob("transforms_train.json")) if Path("./data").exists() else []

for tf in all_transforms:
    parts = list(tf.parts)
    matched = None
    for ds in dataset_ids:
        if ds in parts:
            matched = ds
            break

    if matched is None:
        try:
            idx = parts.index("data")
            matched = parts[idx + 1] if idx + 1 < len(parts) else "unknown"
        except ValueError:
            matched = "unknown"

    scene_roots[matched].add(str(tf.parent))

print("=" * 70)
print("Mapa de scene roots em ./data")
print("=" * 70)
print(f"Total transforms_train.json encontrados: {len(all_transforms)}")
print("")

if not scene_roots:
    print("[vazio] Nenhum scene root encontrado.")
else:
    for ds in sorted(scene_roots.keys()):
        roots = sorted(scene_roots[ds])
        print(f"{ds:20s}: {len(roots)} scene(s)")
        for root in roots[:10]:
            print(f"  - {root}")
        if len(roots) > 10:
            print(f"  ... ({len(roots) - 10} adicionais)")
        print("")

Mapa de scene roots em ./data
Total transforms_train.json encontrados: 0

[vazio] Nenhum scene root encontrado.


In [6]:
# Celula 5.7 - Normalizar estruturas duplicadas (dry-run por padrao)
import json
import shutil
from datetime import datetime, timezone

EXECUTE_NORMALIZE = False
TARGET_DATASETS = DATASETS_TO_INSTALL if "DATASETS_TO_INSTALL" in globals() and DATASETS_TO_INSTALL else None

if TARGET_DATASETS is None:
    try:
        TARGET_DATASETS = [d.item_id for d in catalog.datasets]
    except Exception:
        TARGET_DATASETS = ["blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples", "custom"]


def find_duplicate_nested_dirs(dataset_root: Path):
    pairs = []
    if not dataset_root.exists():
        return pairs

    for parent in dataset_root.rglob("*"):
        if not parent.is_dir():
            continue
        inner = parent / parent.name
        if inner.is_dir():
            pairs.append((parent, inner))

    return pairs


def build_actions(parent: Path, inner: Path, dataset: str):
    actions = []
    for src in sorted(inner.iterdir(), key=lambda p: p.name):
        dst = parent / src.name
        actions.append(
            {
                "dataset": dataset,
                "parent": str(parent),
                "inner": str(inner),
                "src": str(src),
                "dst": str(dst),
                "conflict": dst.exists(),
            }
        )
    return actions


proposed_actions = []
all_pairs = []

for ds in TARGET_DATASETS:
    dataset_root = Path("./data") / ds
    pairs = find_duplicate_nested_dirs(dataset_root)
    for parent, inner in pairs:
        all_pairs.append((ds, parent, inner))
        proposed_actions.extend(build_actions(parent, inner, ds))

print("=" * 70)
print("Normalizacao de datasets: flatten X/X")
print("=" * 70)
print(f"Modo: {'EXECUTE' if EXECUTE_NORMALIZE else 'DRY-RUN'}")
print(f"Datasets alvo: {TARGET_DATASETS}")
print(f"Pares duplicados encontrados: {len(all_pairs)}")
print(f"Acoes propostas: {len(proposed_actions)}")
print("")

if not proposed_actions:
    print("Nenhuma acao necessaria.")
else:
    for i, action in enumerate(proposed_actions, 1):
        marker = "[CONFLICT]" if action["conflict"] else "[OK]"
        print(f"{i:03d}. {marker} {action['src']} -> {action['dst']}")

applied_actions = []
skipped_conflicts = []
errors = []

if EXECUTE_NORMALIZE and proposed_actions:
    for action in proposed_actions:
        src = Path(action["src"])
        dst = Path(action["dst"])

        if action["conflict"]:
            skipped_conflicts.append(action)
            continue

        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            applied_actions.append(action)
        except Exception as exc:
            action_err = dict(action)
            action_err["error"] = str(exc)
            errors.append(action_err)

    # Remove diretorios internos vazios apos move
    for _, _, inner in all_pairs:
        try:
            if inner.exists() and not any(inner.iterdir()):
                inner.rmdir()
        except Exception:
            pass

print("")
print("Resumo:")
print(f"- Aplicadas: {len(applied_actions)}")
print(f"- Conflitos ignorados: {len(skipped_conflicts)}")
print(f"- Erros: {len(errors)}")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
normalize_log_file = ARTIFACTS_DIR / "normalize_log.json"
normalize_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "execute": EXECUTE_NORMALIZE,
    "target_datasets": TARGET_DATASETS,
    "pairs_found": len(all_pairs),
    "proposed": proposed_actions,
    "applied": applied_actions,
    "skipped_conflicts": skipped_conflicts,
    "errors": errors,
}
normalize_log_file.write_text(json.dumps(normalize_log, indent=2), encoding="utf-8")
print(f"Log salvo em: {normalize_log_file}")

Normalizacao de datasets: flatten X/X
Modo: DRY-RUN
Datasets alvo: ['blender_synthetic', 'd_nerf', 'mipnerf360', 'tanks_and_temples', 'custom']
Pares duplicados encontrados: 0
Acoes propostas: 0

Nenhuma acao necessaria.

Resumo:
- Aplicadas: 0
- Conflitos ignorados: 0
- Erros: 0
Log salvo em: artifacts\normalize_log.json


In [ ]:
# Celula 6 - Construir plano de execucao method x dataset
matrix_plan = []
dataset_found_count = 0

# Compatibilidade recomendada por tipo de cena
dataset_type = {
    "blender_synthetic": "static",
    "mipnerf360": "static",
    "tanks_and_temples": "static",
    "d_nerf": "dynamic",
    "custom": "any",
}
method_type = {
    "gs_static": "static",
    "nerf_static": "static",
    "gs_dynamic": "dynamic",
    "nerf_dynamic": "dynamic",
}

compat_skips = 0
for dataset_name in selected_datasets:
    root = choose_scene_root(dataset_name)
    if root is None:
        print(f"[SKIP] {dataset_name}: nao encontrado em ./data")
        continue

    dataset_found_count += 1
    ds_kind = dataset_type.get(dataset_name, "any")

    for method_name in selected_methods:
        mt_kind = method_type.get(method_name, "any")
        if APPLY_COMPATIBILITY_FILTER and ds_kind != "any" and mt_kind != "any" and ds_kind != mt_kind:
            compat_skips += 1
            continue

        matrix_plan.append({
            "dataset": dataset_name,
            "root": str(root),
            "method": method_name,
            "split": "train",
        })

matrix_plan_full = list(matrix_plan)

if RUN_MODE == "quick_check":
    matrix_plan = matrix_plan[:QUICK_CHECK_MAX_COMBOS]

print("")
print(f"Total de datasets encontrados: {dataset_found_count}/{len(selected_datasets)}")
print(f"Total de metodos disponiveis: {len(selected_methods)}")
print(f"Combinacoes apos filtro de compatibilidade: {len(matrix_plan_full)}")
if compat_skips:
    print(f"Combinacoes puladas por incompatibilidade: {compat_skips}")
print(f"Modo de execucao: {RUN_MODE}")
if RUN_MODE == "quick_check":
    print(f"Quick check: {len(matrix_plan)} combinacao(oes), preset={QUICK_CHECK_PRESET}")
else:
    print(f"Execucao full: {len(matrix_plan)} combinacao(oes), preset={PRESET}")

if "gs_dynamic" in selected_methods:
    print("[nota] gs_dynamic nesta base esta em modo stub (execucao muito rapida e metricas de placeholder).")
    print("[nota] Para treino real, prefira gs_static/nerf_* com dependencias third_party completas.")

print("")
for i, item in enumerate(matrix_plan[:10], 1):
    print(f"{i:02d}. {item['dataset']:20s} x {item['method']:15s} -> {item['root']}")
if len(matrix_plan) > 10:
    print("... (mostrando apenas as 10 primeiras)")

if not matrix_plan:
    print("")
    print("ERRO: Nenhuma combinacao valida encontrada.")
    print("")
    print("Opcoes:")
    print("1. Ajuste RUN_MODE para quick_check e QUICK_CHECK_MAX_COMBOS")
    print("2. Revise APPLY_COMPATIBILITY_FILTER")
    print("3. Verifique datasets com a Celula 5 e 5.5")
    print("4. Se necessario, use ONLY_DATASETS/ONLY_METHODS")
    raise RuntimeError("Nenhuma combinacao valida encontrada. Verifique os datasets em ./data.")

In [ ]:
# Cell 6 - Pre-check rapido por combinacao planejada
precheck_rows = []
for item in matrix_plan:
    row = {
        "dataset": item["dataset"],
        "scene_id": item["scene_id"],
        "method": item["method"],
        "root": item["root"],
        "split": item["split"],
        "ok": True,
        "reason": "",
    }
    try:
        dataset_result = run_cmd([
            sys.executable,
            "-m",
            "nvs_benchmark.cli",
            "dataset-check",
            "--dataset",
            item["dataset"],
            "--root",
            item["root"],
            "--split",
            item["split"],
        ], check=False)
        row["ok"] = dataset_result.returncode == 0
        if not row["ok"]:
            row["reason"] = "dataset-check falhou"
    except Exception as exc:
        row["ok"] = False
        row["reason"] = str(exc)
    precheck_rows.append(row)

dump_json(METRICS_DIR / f"{RUN_ID}_precheck.json", precheck_rows)
print("Pre-check concluido para", len(precheck_rows), "combinacoes")

In [ ]:
# Cell 7 - Execucao da matriz com snapshot unico por cena
run_results = []
executed_snapshot: dict[str, dict] = {}

plan_to_run = [item for item in matrix_plan if item["method"] not in set(SKIP_METHODS)]
if RUN_MODE == "quick_check":
    plan_to_run = plan_to_run[:QUICK_CHECK_MAX_COMBOS]

for index, item in enumerate(plan_to_run, start=1):
    combo_key = item["matrix_key"]
    pair_dir = RUNS_DIR / combo_key
    pair_snapshot = METRICS_DIR / f"{combo_key}.json"
    cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "method-run",
        "--method",
        item["method"],
        "--dataset",
        item["dataset"],
        "--root",
        item["root"],
        "--split",
        item["split"],
        "--preset",
        PRESET,
        "--output-dir",
        str(ARTIFACTS_DIR),
        "--log-dir",
        str(LOG_DIR),
        "--compute-metrics",
        "--snapshot-file",
        str(pair_snapshot),
        "--append-snapshot",
        "--strict-results",
        "--min-required-pairs",
        "1",
    ]
    print(f"[{index}/{len(plan_to_run)}] {combo_key}")
    result = run_cmd(cmd, check=False)
    ok = result.returncode == 0
    run_results.append({**item, "ok": ok, "returncode": result.returncode, "snapshot_file": str(pair_snapshot), "log_dir": str(pair_dir)})
    if ok and pair_snapshot.exists():
        try:
            payload = json.loads(pair_snapshot.read_text(encoding="utf-8"))
            if isinstance(payload, dict):
                executed_snapshot[combo_key] = payload.get(item["method"], payload)
        except Exception as exc:
            run_results[-1]["ok"] = False
            run_results[-1]["reason"] = str(exc)

run_results_file = METRICS_DIR / f"{RUN_ID}_run_results.json"
dump_json(run_results_file, run_results)
dump_json(EXECUTED_FILE, executed_snapshot)
print(f"Concluido: {sum(1 for row in run_results if row['ok'])}/{len(run_results)}")

In [ ]:
# Cell 8 - Consolidacao final sem sobrescrever cenas diferentes
consolidated = {}
for item in run_results:
    if not item.get("ok"):
        continue
    key = item["matrix_key"]
    snapshot_file = Path(item["snapshot_file"])
    if not snapshot_file.exists():
        continue
    try:
        payload = json.loads(snapshot_file.read_text(encoding="utf-8"))
    except Exception:
        continue
    if isinstance(payload, dict):
        value = payload.get(item["method"])
        if isinstance(value, dict):
            consolidated[key] = value
        elif payload:
            consolidated[key] = payload

dump_json(SNAPSHOT_FILE, consolidated)
print("Snapshot consolidado:", SNAPSHOT_FILE)
print("Entradas consolidadas:", len(consolidated))

In [ ]:
# Cell 9 - Resumo final e manifestos auxiliares
from collections import Counter

status_counts = Counter("ok" if item.get("ok") else "fail" for item in run_results)
final_status = []
for item in matrix_plan:
    match = next((row for row in run_results if row["matrix_key"] == item["matrix_key"]), None)
    final_status.append({
        "dataset": item["dataset"],
        "scene_id": item["scene_id"],
        "method": item["method"],
        "matrix_key": item["matrix_key"],
        "root": item["root"],
        "status": "ok" if match and match.get("ok") else "fail",
        "returncode": None if not match else match.get("returncode"),
    })

dump_json(METRICS_DIR / f"{RUN_ID}_plan_full.json", matrix_plan)
dump_json(METRICS_DIR / f"{RUN_ID}_final_status.json", final_status)
print("Resumo:", dict(status_counts))
print("Snapshot final:", SNAPSHOT_FILE)
print("Manifestos em:", METRICS_DIR)

In [ ]:
# Cell 10 - Relatorio consolidado
report_cmd = [
    sys.executable,
    "-m",
    "nvs_benchmark.cli",
    "report-generate",
    "--snapshot-file",
    str(SNAPSHOT_FILE),
    "--output-dir",
    str(REPORTS_DIR),
    "--report-name",
    REPORT_NAME,
    "--log-dir",
    str(LOG_DIR),
    "--strict-snapshot",
    "--min-methods",
    "1",
    "--require-finite-metrics",
]
if not GENERATE_PDF:
    report_cmd.append("--no-pdf")

report_result = run_cmd(report_cmd, check=False)
if report_result.returncode != 0:
    raise RuntimeError("Falha ao gerar relatorio consolidado")

REPORT_HTML = REPORTS_DIR / f"{REPORT_NAME}.html"
print("Relatorio HTML:", REPORT_HTML)

In [ ]:
# Cell 11 - Exibir relatorio no notebook
from IPython.display import IFrame, display

if REPORT_HTML.exists():
    display(IFrame(src=str(REPORT_HTML), width=1200, height=720))
else:
    print("Relatorio HTML nao encontrado:", REPORT_HTML)

In [ ]:
# Cell 12 - Compactar artefatos
archive_base = "/content/nvs_benchmark_full_matrix_artifacts"
archive_file = shutil.make_archive(archive_base, "zip", str(REPO_DIR if IN_COLAB else Path.cwd()), "artifacts")
print("Arquivo gerado:", archive_file)

if IN_COLAB:
    from google.colab import files  # type: ignore
    files.download(archive_file)

In [ ]:
# Cell 13 - Backup opcional no Google Drive
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/NVS_Benchmark_Full_Matrix")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    target_artifacts = DRIVE_OUTPUT_DIR / "artifacts"
    if target_artifacts.exists():
        shutil.rmtree(target_artifacts)
    shutil.copytree(ARTIFACTS_DIR, target_artifacts)
    print("Backup concluido em:", target_artifacts)
else:
    print("Backup no Drive desabilitado.")

In [ ]:
# Cell 14 - Resumo final
print("=" * 70)
print("NVS Benchmark - Colab Full Matrix")
print("=" * 70)
print("Run ID:", RUN_ID)
print("Preset:", PRESET)
print("Modo:", RUN_MODE)
print("Plano:", len(matrix_plan))
print("Executados:", sum(1 for item in run_results if item.get("ok")))
print("Snapshot:", SNAPSHOT_FILE)
print("Relatorio:", REPORT_HTML)
print("=" * 70)